# LangChain LangSmith Tracer Reference

# `OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS`

Allowlist of LangSmith-only metadata keys for which the nearest enclosing tracer configuration overrides an inherited value.

```python
OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS: frozenset[str] = frozenset(
    {"ls_agent_type"}
)
```

Other tracer metadata follows first-value-wins propagation so child configurations do not replace values inherited from ancestors.

---

# `log_error_once`

Logs only the first exception of each type for a given method name.

```python
log_error_once(
    method: str, # Name of the method that raised the exception
    exception: Exception, # Exception to log
) -> None
```

Repeated calls with the same `(method, exception type)` pair are ignored, regardless of the exception message.

---

# `wait_for_all_tracers`

Flushes the process-wide LangSmith run-tree client when it has been initialized.

```python
wait_for_all_tracers(
) -> None
```

---

# `get_client`

Returns the cached LangSmith client used by tracing.

```python
get_client(
) -> Client # Cached LangSmith client
```

---





# `LangChainTracer: BaseTracer`

Synchronous tracer that posts LangChain runs to LangSmith and patches them when they finish or fail.

## Fields

```python
run_inline = True # Run callback handling inline
latest_run: Run | None = None # Compact copy of the most recently completed root run
run_has_token_event_map: dict[str, bool] = {} # Whether each model run has recorded a token event
tracing_metadata: dict[str, str] | None # Tracer-level metadata defaults
```

The constructor also initializes the example ID, project name, LangSmith client, and tracer-level tags.

## Constructor

```python
LangChainTracer(
    example_id: UUID | str | None = None, # Example associated with root runs; strings are converted to UUID
    project_name: str | None = None, # LangSmith project; defaults to the configured tracer project
    client: Client | None = None, # LangSmith client; defaults to get_client()
    tags: list[str] | None = None, # Tags merged into traced runs
    *,
    metadata: Mapping[str, str] | None = None, # Tracer-level metadata defaults copied by the constructor
    **kwargs: Any, # Arguments forwarded to BaseTracer
) -> None
```

## Methods

### `copy_with_metadata_defaults`

Creates another tracer with merged tracer-level metadata and tags.

```python
copy_with_metadata_defaults(
    self,
    *,
    metadata: Mapping[str, str] | None = None, # Additional tracer-level metadata defaults
    tags: list[str] | None = None, # Additional tracer-level tags
) -> LangChainTracer # New tracer sharing the current trace bookkeeping
```

Existing metadata values take precedence unless the key is in `OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS`, in which case the new value replaces the inherited value. Supplied tags are combined with existing tags, deduplicated, and sorted.

The new tracer keeps the same example ID, project, client, run map, ordering map, and externally managed run-ID set.

### `on_chat_model_start`

Creates, starts, and persists a chat-model run.

```python
on_chat_model_start(
    self,
    serialized: dict[str, Any], # Serialized chat model
    messages: list[list[BaseMessage]], # Batches of input messages
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run data
) -> Run # Created chat-model run
```

Messages are serialized with `dumpd()` and stored under `inputs["messages"]`. The created run uses run type `"llm"`, receives a UTC start event, and is passed through the normal trace-start and chat-model-start hooks.

### `get_run_url`

Returns the LangSmith URL for the most recently completed root run.

```python
get_run_url(
    self,
) -> str # LangSmith root-run URL
```

The client lookup is retried up to five times for `LangSmithError` using exponential jitter.

Raises `ValueError` when no completed root run has been recorded or when a run URL cannot be returned.

### `wait_for_futures`

Flushes pending work through the configured LangSmith client.

```python
wait_for_futures(
    self,
) -> None
```

In [ ]:
import os # Import os to configure environment variables
from getpass import getpass # Import getpass to enter the API key securely

os.environ["LANGSMITH_TRACING"] = "true" # Enable LangSmith tracing
os.environ["LANGSMITH_API_KEY"] = getpass("Enter your LangSmith API key: ") # Enter the key securely
os.environ["LANGSMITH_PROJECT"] = "langchain-tracer-demo" # Set the LangSmith project name

from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.tracers.langchain import LangChainTracer # Import the real tracer


tracer = LangChainTracer( # Create the real LangSmith tracer
    project_name="langchain-tracer-demo", # Send traces to this project
    tags=["jupyter", "demo"], # Add default tags
    metadata={"source": "notebook"}, # Add default metadata
) # Finish creating the tracer

child_tracer = tracer.copy_with_metadata_defaults( # Create a tracer with additional defaults
    tags=["math"], # Add another tag
    metadata={"operation": "square"}, # Add more metadata
) # Finish creating the copied tracer


def square(number: int) -> int: # Define the operation to trace
    return number * number # Return the squared value


square_runnable = RunnableLambda(square) # Convert the function into a LangChain runnable

result = square_runnable.invoke( # Execute and trace the operation
    5, # Provide the runnable input
    config={ # Configure this run
        "callbacks": [child_tracer], # Attach the real LangSmith tracer
        "run_name": "square_operation", # Set the run name
        "tags": ["calculation"], # Add run-specific tags
        "metadata": {"input_type": "integer"}, # Add run-specific metadata
    },
) # Finish invoking the runnable

child_tracer.wait_for_futures() # Wait until the trace is uploaded

print("Result:", result) # Display the result
print("Latest run ID:", child_tracer.latest_run.id) # Display the completed run ID
print("LangSmith run URL:", child_tracer.get_run_url()) # Display the real trace URL

## Behaviour

The tracer assigns its project name as the run session, merges tracer tags with run tags, and assigns its client to runs that do not already have one. When the active LangSmith tracing context is explicitly disabled, runs are marked so posting and patching are skipped.

Root LLM, chat-model, chain, tool, and retriever runs receive `example_id` as their reference example ID. Runs are posted when they start and patched when they finish or fail. Chains with deferred inputs are instead posted when their realized inputs become available at completion or error time.

Only the first streamed token event is retained for each model run, and the associated generation chunk is not stored.

On model completion, usage metadata found in serialized generation messages is aggregated and placed in the run metadata before the run is patched.

When a root run completes, `latest_run` stores a compact copy that excludes the nested child-run tree while retaining its inputs and outputs.